# SAC Calibration — Kaggle free-GPU run (v2)

Mirrors `Final/SAC/run_sac_calib.pbs` (cluster version): unit tests, then a
Hard-variant 1-seed / 50-epoch smoke+calibration run.

v2: adds a minimal `finrl` shim (only `YahooDownloader` + `FeatureEngineer`,
backed by yfinance + stockstats) — the full finrl package isn't installed on
Kaggle and pulling it in breaks the preinstalled environment. The shim is
calibration-grade: functionally equivalent, not byte-identical to the
cluster's finrl preprocessing, so paper-table numbers still come from the
cluster.

**Kaggle setup (right sidebar) before running:**
1. Accelerator: **GPU T4 x2** (or P100)
2. Internet: **On** (yfinance downloads data)

Then **Save Version → Save & Run All (Commit)** — runs server-side, laptop can be off.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable GPU in settings!')

In [ ]:
%cd /kaggle/working
!rm -rf Regime-Aware-RL-URECA-
!git clone -q --depth 1 https://github.com/Bharath21234/Regime-Aware-RL-URECA-.git
%cd Regime-Aware-RL-URECA-/Final/SAC
!pip install -q yfinance hmmlearn stockstats

In [ ]:
# finrl shim: the repo imports YahooDownloader/FeatureEngineer from finrl,
# which lives in the cluster venv. Recreate just those two classes.
import os
base = '/kaggle/working/shim/finrl/meta/preprocessor'
os.makedirs(base, exist_ok=True)
for d in ['/kaggle/working/shim/finrl', '/kaggle/working/shim/finrl/meta', base]:
    open(os.path.join(d, '__init__.py'), 'w').close()

yd = '''
import pandas as pd
import yfinance as yf

class YahooDownloader:
    def __init__(self, start_date, end_date, ticker_list):
        self.start_date, self.end_date = start_date, end_date
        self.ticker_list = ticker_list

    def fetch_data(self):
        frames = []
        for tic in self.ticker_list:
            d = yf.download(tic, start=self.start_date, end=self.end_date,
                            auto_adjust=True, progress=False)
            if d is None or d.empty:
                continue
            if isinstance(d.columns, pd.MultiIndex):
                d.columns = d.columns.get_level_values(0)
            d = d.reset_index()
            d.columns = [str(c).lower() for c in d.columns]
            d['tic'] = tic
            d['date'] = pd.to_datetime(d['date']).dt.strftime('%Y-%m-%d')
            frames.append(d[['date', 'open', 'high', 'low', 'close', 'volume', 'tic']])
        return (pd.concat(frames, ignore_index=True)
                  .sort_values(['date', 'tic']).reset_index(drop=True))
'''

pp = '''
import pandas as pd
from stockstats import StockDataFrame as Sdf

class FeatureEngineer:
    def __init__(self, use_technical_indicator=True, tech_indicator_list=None,
                 use_vix=False, use_turbulence=False, user_defined_feature=False):
        self.tech_indicator_list = tech_indicator_list or []

    def preprocess_data(self, df):
        out = []
        for tic, g in df.copy().sort_values(['tic', 'date']).groupby('tic'):
            g = g.reset_index(drop=True)
            s = Sdf.retype(g.copy())
            for ind in self.tech_indicator_list:
                g[ind] = s[ind].values
            out.append(g)
        # NaN warmup rows are left in: the caller does dropna() itself,
        # matching the cluster pipeline's behaviour.
        return (pd.concat(out, ignore_index=True)
                  .sort_values(['date', 'tic']).reset_index(drop=True))
'''

open(os.path.join(base, 'yahoodownloader.py'), 'w').write(yd)
open(os.path.join(base, 'preprocessors.py'), 'w').write(pp)

# sanity: shim imports and downloads one ticker
import sys
sys.path.insert(0, '/kaggle/working/shim')
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
t = YahooDownloader('2023-01-01', '2023-02-01', ['SPY']).fetch_data()
print('shim OK —', len(t), 'rows,', list(t.columns))

In [ ]:
# Step 1: unit tests (no finrl needed, but PYTHONPATH is harmless here)
!PYTHONPATH=/kaggle/working/shim python -u tests_sac.py

In [ ]:
# Step 2: calibration run — Hard variant, 1 seed, 50 epochs, mv reward.
import time
t0 = time.time()
!PYTHONPATH=/kaggle/working/shim python -u run_sac.py --variant hard --seeds 1 --epochs 50 --reward_mode mv --calib
dt = time.time() - t0
print(f'\nTotal calib wall-clock: {dt/3600:.2f} h'
      f' => per-epoch {dt/50/60:.1f} min'
      f' => a full 300-epoch seed on this GPU ~ {dt/3600*6:.1f} h')

In [ ]:
# Step 3: package results for download (Output tab of the saved version)
import glob, json, os, shutil
if os.path.isdir('results'):
    for f in glob.glob('results/**/seed_*.json', recursive=True):
        print(f, '->', json.load(open(f)))
    shutil.make_archive('/kaggle/working/sac_calib_results', 'zip', 'results')
    print('\nSaved /kaggle/working/sac_calib_results.zip')
else:
    print('No results/ dir — the calibration run failed; check the cell above.')